Import Libraries

In [1]:
import pandas as pd
import numpy as np

Load Dataset

In [5]:
df = pd.read_excel("Final Mon-2 Dataset.xlsx")

Display Dataset

In [6]:
df.head()

,Invoice No,Stock Code,Description,Quantity,Invoice Date,Invoice Time,UnitPrice,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-01-12,08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-01-12,08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-01-12,08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-01-12,08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-01-12,08:26:00,3.39,17850.0,United Kingdom


In [7]:
df.shape

(541908, 9)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541908 entries, 0 to 541907
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Invoice No    541908 non-null  object        
 1   Stock Code    541908 non-null  object        
 2   Description   540454 non-null  str           
 3   Quantity      541908 non-null  int64         
 4   Invoice Date  232959 non-null  datetime64[us]
 5   Invoice Time  232959 non-null  object        
 6   UnitPrice     541908 non-null  float64       
 7   Customer ID   406828 non-null  float64       
 8   Country       541908 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(2)
memory usage: 37.2+ MB


In [9]:
df.dtypes

Invoice No              object
Stock Code              object
Description                str
Quantity                 int64
Invoice Date    datetime64[us]
Invoice Time            object
UnitPrice              float64
Customer ID            float64
Country                    str
dtype: object

In [10]:
df.describe()

,Quantity,Invoice Date,UnitPrice,Customer ID
count,541908.000000,232959,541908.000000,406828.000000
mean,9.552245,2011-05-13 13:39:45.829266,4.611120,15287.687443
min,-80995.000000,2010-01-12 00:00:00,-11062.060000,12346.000000
25%,1.000000,2011-03-04 00:00:00,1.250000,13953.000000
50%,3.000000,2011-06-09 00:00:00,2.080000,15152.000000
75%,10.000000,2011-09-06 00:00:00,4.130000,16791.000000
max,80995.000000,2011-12-10 00:00:00,38970.000000,18287.000000
std,218.081359,NaN,96.759942,1713.601248


Missing Values

In [11]:
df.isnull().sum()

Invoice No           0
Stock Code           0
Description       1454
Quantity             0
Invoice Date    308949
Invoice Time    308949
UnitPrice            0
Customer ID     135080
Country              0
dtype: int64

In [14]:
df = df.dropna(subset=['Customer ID'])

Remove Duplicate Records

In [15]:
df = df.drop_duplicates()

Convert Invoice Date Correctly

In [16]:
df['Invoice Date'] = pd.to_datetime(df['Invoice Date'], errors='coerce')

In [17]:
df['Invoice Date'].dtype

dtype('<M8[us]')

Remove Invalid Dates

In [18]:
df = df.dropna(subset=['Invoice Date'])

In [19]:
df['Invoice Date'].isnull().sum()

np.int64(0)

Keep Only Valid Transactions

In [20]:
df = df[df['Quantity'] > 0]

In [21]:
df = df[df['UnitPrice'] > 0]

Create Total Amount

In [22]:
df['Total Amount'] = df['Quantity'] * df['UnitPrice']

In [23]:
df[['Quantity','UnitPrice','Total Amount']].head()

,Quantity,UnitPrice,Total Amount
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


Latest Invoice Date

In [24]:
latest_date = df['Invoice Date'].max()

print(latest_date)

2011-12-10 00:00:00


Calculate Recency

In [25]:
recency = df.groupby('Customer ID')['Invoice Date'].max().reset_index()

recency['Recency'] = (latest_date - recency['Invoice Date']).dt.days

recency = recency[['Customer ID','Recency']]

In [26]:
recency.head()

,Customer ID,Recency
0,12347.0,95
1,12348.0,220
2,12350.0,311
3,12352.0,274
4,12355.0,96


In [27]:
recency.isnull().sum()

Customer ID    0
Recency        0
dtype: int64

Calculate Frequency

In [28]:
frequency = df.groupby('Customer ID')['Invoice No'].nunique().reset_index()

frequency.columns = ['Customer ID','Frequency']

In [29]:
frequency.head()

,Customer ID,Frequency
0,12347.0,5
1,12348.0,1
2,12350.0,1
3,12352.0,3
4,12355.0,1


Calculate Monetary

In [30]:
monetary = df.groupby('Customer ID')['Total Amount'].sum().reset_index()

monetary.columns = ['Customer ID','Monetary']

In [31]:
monetary.head()

,Customer ID,Monetary
0,12347.0,2540.29
1,12348.0,367.00
2,12350.0,334.40
3,12352.0,1296.38
4,12355.0,459.40


Merge RFM

In [32]:
rfm = recency.merge(frequency,on='Customer ID')

rfm = rfm.merge(monetary,on='Customer ID')

View RFM

In [33]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12347.0,95,5,2540.29
1,12348.0,220,1,367.00
2,12350.0,311,1,334.40
3,12352.0,274,3,1296.38
4,12355.0,96,1,459.40


Check Missing Values

In [34]:
rfm.isnull().sum()

Customer ID    0
Recency        0
Frequency      0
Monetary       0
dtype: int64

Summary

In [35]:
rfm.describe()

,Customer ID,Recency,Frequency,Monetary
count,2997.000000,2997.000000,2997.000000,2997.000000
mean,15291.891892,154.419086,2.656323,1289.777471
std,1722.604761,140.146976,4.008919,5273.011876
min,12347.000000,0.000000,1.000000,5.900000
25%,13809.000000,39.000000,1.000000,244.410000
50%,15251.000000,124.000000,2.000000,466.290000
75%,16791.000000,218.000000,3.000000,1064.280000
max,18287.000000,697.000000,106.000000,168469.600000


Save RFM Dataset

In [36]:
rfm.to_csv("RFM_Dataset.csv",index=False)

Final Output

In [37]:
rfm.head(10)

,Customer ID,Recency,Frequency,Monetary
0,12347.0,95,5,2540.29
1,12348.0,220,1,367.00
2,12350.0,311,1,334.40
3,12352.0,274,3,1296.38
4,12355.0,96,1,459.40
5,12356.0,128,1,481.46
6,12357.0,182,1,6207.67
7,12358.0,3,2,1168.06
8,12359.0,9,3,3495.73
9,12362.0,30,5,2135.79
